# Mid Submission Reviewer Walkthrough

This notebook provides a clean reviewer-facing walkthrough of the Version 1 baseline for the Device Protection Claims Triage capstone project.

It is intentionally lightweight. It does not rebuild embeddings, call OpenAI, download models, or modify project data. It reads the generated mid-submission artifacts and summarises the solution flow.

Detailed implementation notebooks remain available separately:

- `notebooks/05_sop_rag_retrieval.ipynb`
- `notebooks/06_workflow_evaluation.ipynb`

## 1. Project Summary

This project implements a rule-grounded Agentic AI decision-support workflow for device protection claim triage.

The system combines:

- Deterministic policy and eligibility rules
- Controlled follow-up selection
- LangGraph workflow orchestration
- Controlled RAG over approved SOP / KB documents
- FAISS persisted semantic retrieval
- Optional cross-encoder reranking
- Analyst guidance formatting
- Agent content safety and response authority guardrails

Core design principle:

**Deterministic rules are authoritative. RAG is non-authoritative and analyst-facing only.**

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

print(f"Project root: {PROJECT_ROOT}")
assert (PROJECT_ROOT / "README.md").exists(), "README.md not found. Run from project root or notebooks folder."

Project root: /Users/sadiqsheik/Documents/MS/IIITB_Course Docs/Capstone Project/BYOC/DP_claims_triage


## 2. Repository Readiness Check

The following check confirms that the key reviewer-facing files and generated artifacts are present.

In [2]:
required_paths = [
    "README.md",
    "requirements.txt",
    ".env.example",
    "docs/mid_submission_summary.md",
    "docs/evaluation_summary.md",
    "docs/architecture_decisions.md",
    "docs/mid_submission_checklist.md",
    "data/artifacts/rag/faiss_semantic_index_v1",
    "data/evaluation/retrieval/retrieval_summary_metrics_with_reranker_v1.csv",
    "data/evaluation/workflow/workflow_development_summary_metrics_v1.csv",
    "data/evaluation/workflow/workflow_development_mismatch_analysis_v1.csv",
    "data/evaluation/safety/safety_adversarial_summary_metrics_v1.csv",
]

readiness = []
for rel_path in required_paths:
    path = PROJECT_ROOT / rel_path
    readiness.append({
        "path": rel_path,
        "exists": path.exists(),
        "type": "directory" if path.is_dir() else "file" if path.exists() else "missing",
    })

readiness_df = pd.DataFrame(readiness)
display(readiness_df)

assert readiness_df["exists"].all(), "One or more required reviewer artifacts are missing."

,path,exists,type
0,README.md,True,file
1,requirements.txt,True,file
2,.env.example,True,file
3,docs/mid_submission_summary.md,True,file
4,docs/evaluation_summary.md,True,file
5,docs/architecture_decisions.md,True,file
6,docs/mid_submission_checklist.md,True,file
7,data/artifacts/rag/faiss_semantic_index_v1,True,directory
8,data/evaluation/retrieval/retrieval_summary_me...,True,file
9,data/evaluation/workflow/workflow_development_...,True,file


## 3. Baseline Architecture Flow

High-level workflow:

Claim Intake
  → Deterministic Triage
  → Controlled Follow-up Selection
  → Optional Controlled RAG Retrieval
  → Optional Cross-Encoder Reranking
  → Analyst Guidance Formatter
  → Agent Content Safety Guardrail
  → Response Authority Guardrail
  → Protected Final Response

The deterministic triage layer is responsible for authoritative routing decisions.  
RAG is used only to provide analyst-facing SOP / KB guidance.

## 4. FAISS Semantic Index Manifest

The persisted semantic index supports reproducible controlled retrieval over the approved SOP / KB corpus.

In [3]:
manifest_path = PROJECT_ROOT / "data/artifacts/rag/faiss_semantic_index_v1/semantic_index_manifest.json"

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

manifest_fields = [
    "index_type",
    "embedding_model",
    "vector_dimension",
    "chunk_count",
    "corpus_fingerprint",
    "chunk_order_fingerprint",
]

manifest_view = {
    field: manifest.get(field, "<not present>")
    for field in manifest_fields
}

display(pd.DataFrame(manifest_view.items(), columns=["field", "value"]))

,field,value
0,index_type,IndexFlatIP
1,embedding_model,text-embedding-3-small
2,vector_dimension,<not present>
3,chunk_count,<not present>
4,corpus_fingerprint,f73f827d3614e2987a7e976c687043f3fe9e2f5faeeb9c...
5,chunk_order_fingerprint,0a0912f687603e11336f82357d7d5db01d94f2b719b1fb...


## 5. Retrieval Evaluation Summary

This section reads the generated retrieval evaluation artifact.

Retrieval methods evaluated:

- Lexical TF-IDF
- Semantic Embedding
- Hybrid RRF
- Semantic + Cross-Encoder Reranker

In [4]:
retrieval_summary_path = PROJECT_ROOT / "data/evaluation/retrieval/retrieval_summary_metrics_with_reranker_v1.csv"
retrieval_summary = pd.read_csv(retrieval_summary_path)

display(retrieval_summary)

,method_name,query_count,top_k,hit_at_1,hit_at_3,mrr_at_3,no_result_rate
0,Lexical TF-IDF,14,3,0.571429,0.857143,0.702381,0.0
1,Semantic Embedding,14,3,0.785714,0.928571,0.857143,0.0
2,Hybrid RRF,14,3,0.714286,0.928571,0.797619,0.0
3,Semantic + Cross-Encoder Reranker,14,3,0.785714,0.928571,0.845238,0.0


## 6. Workflow Evaluation Summary

This section reads the generated workflow development evaluation artifact.

The workflow evaluation verifies that the LangGraph workflow completes claim triage and preserves deterministic authority in the protected final response.

In [5]:
workflow_summary_path = PROJECT_ROOT / "data/evaluation/workflow/workflow_development_summary_metrics_v1.csv"
workflow_summary = pd.read_csv(workflow_summary_path)

display(workflow_summary)

,metric,value
0,claim_count,165.000000
1,workflow_completion_rate,1.000000
2,disposition_agreement,0.915152
3,primary_rule_exact_agreement,0.890909
4,primary_rule_acceptable_agreement,0.915152
5,final_response_matches_deterministic_outcome,1.000000
6,final_response_matches_deterministic_rule,1.000000
7,authority_guardrail_aligned_rate,1.000000
8,follow_up_exact_match_rate,0.993939
9,elapsed_seconds,1.882030


## 7. Workflow Mismatch Review

The workflow reached strong development-label agreement, while preserving deterministic outputs in every case.

Remaining mismatches are treated as deterministic rule/data improvement opportunities rather than LangGraph or RAG failures.

In [6]:
mismatch_path = PROJECT_ROOT / "data/evaluation/workflow/workflow_development_mismatch_analysis_v1.csv"
mismatch_df = pd.read_csv(mismatch_path)

print(f"Mismatch rows: {len(mismatch_df)}")

display(mismatch_df.head(10))

Mismatch rows: 15


,claim_id,gold_disposition,predicted_disposition,gold_primary_rule_id,gold_acceptable_primary_rule_ids,predicted_triggering_rule_id,disposition_match,primary_rule_exact_match,primary_rule_acceptable_match,gold_follow_up_question_ids,predicted_follow_up_question_ids,follow_up_exact_match,mismatch_type,gold_rule_family,predicted_rule_family,known_baseline_limitation
0,CLM-000118,INFO_REQUIRED,INFO_REQUIRED,EVD-001,['EVD-001'],EVD-001,True,True,True,['FUP-012'],['FUP-011'],False,FOLLOW_UP_SELECTION_MISMATCH,EVD,EVD,False
1,CLM-000154,MANUAL_REVIEW,NOT_ELIGIBLE,EVD-002,['EVD-002'],COV-001,False,False,False,[],[],True,DISPOSITION_OR_RULE_MISMATCH,EVD,COV,False
2,CLM-000156,MANUAL_REVIEW,NOT_ELIGIBLE,EVD-002,['EVD-002'],COV-001,False,False,False,[],[],True,DISPOSITION_OR_RULE_MISMATCH,EVD,COV,False
3,CLM-000157,MANUAL_REVIEW,NOT_ELIGIBLE,EVD-002,['EVD-002'],COV-001,False,False,False,[],[],True,DISPOSITION_OR_RULE_MISMATCH,EVD,COV,False
4,CLM-000170,MANUAL_REVIEW,PROCEED,DATA-002,['DATA-002'],OUT-001,False,False,False,[],[],True,DISPOSITION_OR_RULE_MISMATCH,DATA,OUT,False
5,CLM-000171,MANUAL_REVIEW,PROCEED,DATA-002,['DATA-002'],OUT-001,False,False,False,[],[],True,DISPOSITION_OR_RULE_MISMATCH,DATA,OUT,False
6,CLM-000172,MANUAL_REVIEW,PROCEED,DATA-002,['DATA-002'],OUT-001,False,False,False,[],[],True,DISPOSITION_OR_RULE_MISMATCH,DATA,OUT,False
7,CLM-000173,MANUAL_REVIEW,PROCEED,DATA-002,['DATA-002'],OUT-001,False,False,False,[],[],True,DISPOSITION_OR_RULE_MISMATCH,DATA,OUT,False
8,CLM-000176,MANUAL_REVIEW,PROCEED,EXC-002,['EXC-002'],OUT-001,False,False,False,[],[],True,DISPOSITION_OR_RULE_MISMATCH,EXC,OUT,True
9,CLM-000177,MANUAL_REVIEW,PROCEED,EXC-002,['EXC-002'],OUT-001,False,False,False,[],[],True,DISPOSITION_OR_RULE_MISMATCH,EXC,OUT,True


## 8. Safety and Adversarial Evaluation Summary

This section reads the generated safety evaluation artifact.

The safety evaluation checks whether unsafe proposed agent content is blocked and whether deterministic outputs are preserved.

In [7]:
safety_summary_path = PROJECT_ROOT / "data/evaluation/safety/safety_adversarial_summary_metrics_v1.csv"
safety_summary = pd.read_csv(safety_summary_path)

display(safety_summary)

,metric,value
0,test_case_count,24.0
1,critical_safety_pass_rate,1.0
2,deterministic_outcome_preservation_rate,1.0
3,deterministic_rule_preservation_rate,1.0
4,unsafe_override_block_rate,1.0
5,mechanical_prohibited_behavior_rate,0.0


## 9. Safety Boundary

The system does not:

- Approve claims
- Deny claims finally
- Determine fraud
- Make payout decisions
- Override deterministic policy eligibility
- Use RAG to change claim routing
- Generate uncontrolled customer follow-up wording

The system does:

- Provide decision-support recommendations
- Select approved follow-up questions
- Retrieve approved SOP / KB guidance for analysts
- Preserve deterministic authority in the final response
- Block unsafe agent-generated overrides

## 10. Reproducibility Command

Full regression can be run from the project root using:

    python -m unittest discover -s tests -p "test_*.py" -v

Current expected result:

    136 tests passed

Live semantic retrieval requires a local `.env` file created from `.env.example`.
The actual `.env` file must not be committed.

## 11. Final Submission Plan

Planned final-submission activities:

- Run locked held-out evaluation.
- Review deterministic mismatch analysis.
- Make only justified rule/data refinements.
- Refresh evaluation artifacts if logic changes.
- Prepare final report and presentation.
- Add concise demo walkthrough.
- Complete final repository hygiene and access checks.

This mid-submission repository represents a functional, modular, and evaluated Version 1 baseline.